In [ ]:
from pathlib import Path
import os
import sys

_candidates = [Path.cwd(), Path.cwd().parent, Path.cwd() / "cifar10"]
_dataset_root = next(p.resolve() for p in _candidates if (p / "baseline_cifar10.py").exists())
sys.path.insert(0, str(_dataset_root))
os.chdir(_dataset_root)

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import tensorflow as tf
import torch
from torch import nn
from torch.utils.data import DataLoader
from torchvision.datasets import CIFAR10

from baseline_cifar10 import cifar10_transform
from tuned_experiment import TUNED_CHECKPOINT, load_tuned_model

In [ ]:
test_dataset = CIFAR10("data", train=False, download=True, transform=cifar10_transform())
test_loader = DataLoader(test_dataset, batch_size=64, shuffle=False)
torch_model = load_tuned_model().eval()

In [ ]:
class LiteCNN(tf.Module):
    def __init__(self, model):
        super().__init__()
        self.layers = []
        for index, layer in enumerate(model.features):
            if isinstance(layer, nn.Conv2d):
                bn = model.features[index + 1]
                scale = bn.weight.detach() / (bn.running_var + bn.eps).sqrt()
                kernel = layer.weight.detach() * scale.reshape(-1, 1, 1, 1)
                bias = bn.bias.detach() - bn.running_mean * scale
                self.layers.append(("conv", tf.constant(kernel.permute(2, 3, 1, 0).numpy()),
                                    tf.constant(bias.numpy())))
            elif isinstance(layer, nn.ReLU):
                self.layers.append(("relu", None, None))
            elif isinstance(layer, nn.MaxPool2d):
                self.layers.append(("pool", None, None))
        head = model.classifier
        weights = head.weight.detach().reshape(head.out_features, -1, 4, 4)
        weights = weights.permute(2, 3, 1, 0).reshape(head.in_features, head.out_features)
        self.head_weights = tf.constant(weights.numpy())
        self.head_bias = tf.constant(head.bias.detach().numpy())

    @tf.function(input_signature=[tf.TensorSpec([None, 32, 32, 3], tf.float32)])
    def __call__(self, images):
        for kind, kernel, bias in self.layers:
            if kind == "conv":
                images = tf.nn.bias_add(tf.nn.conv2d(images, kernel, strides=1, padding="SAME"), bias)
            elif kind == "relu":
                images = tf.nn.relu(images)
            else:
                images = tf.nn.max_pool2d(images, ksize=2, strides=2, padding="VALID")
        flat = tf.reshape(images, [-1, self.head_weights.shape[0]])
        return tf.linalg.matmul(flat, self.head_weights) + self.head_bias

model = LiteCNN(torch_model)

In [ ]:
# Verify that folding BatchNorm and changing layout preserve the tuned model's logits.
images, _ = next(iter(test_loader))
with torch.inference_mode():
    expected = torch_model(images).numpy()
actual = model(images.permute(0, 2, 3, 1).numpy()).numpy()
np.testing.assert_allclose(actual, expected, atol=1e-4, rtol=1e-4)

In [ ]:
def evaluate_lite(contents):
    interpreter = tf.lite.Interpreter(model_content=contents)
    input_index = interpreter.get_input_details()[0]["index"]
    shape = None
    correct = 0
    for images, labels in test_loader:
        batch = images.permute(0, 2, 3, 1).contiguous().numpy()
        if batch.shape != shape:
            shape = batch.shape
            interpreter.resize_tensor_input(input_index, shape, strict=True)
            interpreter.allocate_tensors()
        interpreter.set_tensor(input_index, batch)
        interpreter.invoke()
        logits = interpreter.get_tensor(interpreter.get_output_details()[0]["index"])
        correct += int((logits.argmax(axis=1) == labels.numpy()).sum())
    return 100 * correct / len(test_dataset)

In [ ]:
baseline_size = TUNED_CHECKPOINT.stat().st_size / 1024
Path("models").mkdir(exist_ok=True)

In [ ]:
converter = tf.lite.TFLiteConverter.from_concrete_functions([model.__call__.get_concrete_function()], model)
tflite_model = converter.convert()
Path("models/tuned_cifar10.tflite").write_bytes(tflite_model)
tflite_size = len(tflite_model) / 1024

In [ ]:
converter = tf.lite.TFLiteConverter.from_concrete_functions([model.__call__.get_concrete_function()], model)
converter.optimizations = [tf.lite.Optimize.DEFAULT]
quantized_model = converter.convert()
Path("models/tuned_cifar10_dynamic.tflite").write_bytes(quantized_model)
quantized_size = len(quantized_model) / 1024

In [ ]:
# Compare model sizes

models = ["Tuned PyTorch", "TFLite", "Quantized TFLite"]
sizes = [baseline_size, tflite_size, quantized_size]

plt.figure(figsize=(8,5))
plt.bar(models, sizes)
plt.ylabel("Model Size (KiB)")
plt.title("Model Size Comparison")
plt.show()

In [ ]:
comparison_df = pd.DataFrame({
    "Model": ["Tuned PyTorch", "TFLite", "Dynamic-range TFLite"],
    "Model size (KiB)": [baseline_size, tflite_size, quantized_size],
})
comparison_df

In [ ]:
accuracy_df = pd.DataFrame({
    "Model": ["TFLite", "Dynamic-range TFLite"],
    "Test accuracy (%)": [evaluate_lite(tflite_model), evaluate_lite(quantized_model)],
})
accuracy_df